# 🐛 CoBug AI Learning Platform — Fixed Colab Runner

**Run cells top to bottom. Only Cell 4 needs your API keys.**

### Fix from original notebook:
Instead of running two Node.js dev servers (which crashes Colab's free RAM),
this notebook **builds the frontend once** and serves it from the backend — only one process runs.

---
**You need:**
- Groq API key → [console.groq.com](https://console.groq.com) (free)
- ngrok auth token → [dashboard.ngrok.com](https://dashboard.ngrok.com) (free)

In [ ]:
# ── Cell 1: Install Node.js 18 ──────────────────────────────────────────────
import subprocess
print('Installing Node.js 18...')
!curl -fsSL https://deb.nodesource.com/setup_18.x | bash - 2>/dev/null
!apt-get install -y nodejs 2>/dev/null | grep -E 'nodejs|already'
node_ver = subprocess.check_output(['node', '--version']).decode().strip()
npm_ver  = subprocess.check_output(['npm',  '--version']).decode().strip()
print(f'Node: {node_ver}  |  npm: {npm_ver}')
print('✅ Node.js ready')

In [ ]:
# ── Cell 2: Install pyngrok ──────────────────────────────────────────────────
!pip install pyngrok -q
print('✅ pyngrok installed')

In [ ]:
# ── Cell 3: Clone the repo ───────────────────────────────────────────────────
import os

REPO_DIR = 'CoBug-ai-learning-platform'

if os.path.exists(REPO_DIR):
    print('Repo already exists, pulling latest changes...')
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/Arjun-115/CoBug-ai-learning-platform.git

os.chdir(REPO_DIR)
print('✅ Working directory:', os.getcwd())

In [ ]:
# ── Cell 4: 🔑 ENTER YOUR API KEYS HERE ─────────────────────────────────────

GROQ_API_KEY    = "gsk_..."   # Required — get from console.groq.com
NGROK_AUTH_TOKEN = "..."      # Required — get from dashboard.ngrok.com
GITHUB_TOKEN    = ""          # Optional — raises GitHub API rate limit

# Basic validation
assert GROQ_API_KEY.startswith('gsk_'), '❌ GROQ_API_KEY looks wrong — should start with gsk_'
assert len(NGROK_AUTH_TOKEN) > 10,      '❌ NGROK_AUTH_TOKEN looks empty or wrong'
print('✅ Keys look good')

In [ ]:
# ── Cell 5: Install dependencies & build frontend ────────────────────────────
# This takes 2-4 minutes — wait for the ✅ messages

print('📦 Installing backend dependencies...')
!cd backend && npm install --legacy-peer-deps 2>&1 | tail -3
print('✅ Backend deps done\n')

print('📦 Installing frontend dependencies...')
!cd frontend && npm install --legacy-peer-deps 2>&1 | tail -3
print('✅ Frontend deps done\n')

# Fix: postcss.config.js uses ES module syntax but Node loads it as CJS
import re
postcss_path = 'frontend/postcss.config.js'
with open(postcss_path, 'r') as f:
    content = f.read()
if 'export default' in content:
    content = content.replace('export default', 'module.exports =')
    with open(postcss_path, 'w') as f:
        f.write(content)
    print('✅ Fixed postcss.config.js (ES module → CommonJS)')
else:
    print('✅ postcss.config.js already OK')

print('🏗️  Building frontend (1-2 min)...')
!cd frontend && npm run build 2>&1

import os
if os.path.exists('frontend/dist/index.html'):
    print('\n✅ Frontend built successfully!')
else:
    print('\n❌ Build failed — check the error above')

In [ ]:
# ── Cell 6: Configure backend → Launch ──────────────────────────────────────
# KEY FIX: only ONE Node.js process runs (backend serves the built frontend)

import os, subprocess, shutil, time
from pyngrok import ngrok

# ── Step 1: Write .env ──────────────────────────────────────────────────────
env_lines = [f'GROQ_API_KEY={GROQ_API_KEY}', 'PORT=5000']
if GITHUB_TOKEN:
    env_lines.append(f'GITHUB_TOKEN={GITHUB_TOKEN}')
with open('backend/.env', 'w') as f:
    f.write('\n'.join(env_lines) + '\n')
print('✅ .env written')

# ── Step 2: Copy frontend build into backend/public ─────────────────────────
if os.path.exists('backend/public'):
    shutil.rmtree('backend/public')
shutil.copytree('frontend/dist', 'backend/public')
print('✅ Frontend dist copied to backend/public')

# ── Step 3: Patch server.js ─────────────────────────────────────────────────
# Insert static middleware RIGHT AFTER app is created (before any routes),
# so it takes priority over any existing root '/' route that returns JSON.
with open('backend/server.js', 'r') as f:
    server_code = f.read()

# Remove old patch if re-running
if 'added by Colab runner' in server_code:
    import re
    server_code = re.sub(
        r'\n// ── Serve frontend.*?\}\);\n',
        '', server_code, flags=re.DOTALL
    )
    print('♻️  Removed old patch')

static_block = """
// ── Serve frontend build (added by Colab runner) ──
const path = require('path');
app.use(express.static(path.join(__dirname, 'public')));
app.get('/', (req, res) => {
  res.sendFile(path.join(__dirname, 'public', 'index.html'));
});
"""

# Insert AFTER the first app.use() call (i.e. right after express setup)
# This ensures static files are served before any API route catches '/'
import re
# Find the first app.use( line and insert after it
patched = re.sub(
    r'(app\.use\([^)]*\);)',
    r'\1' + static_block,
    server_code, count=1
)
# Also replace any existing root GET that returns API status
patched = re.sub(
    r"app\.get\(['\"/]['\"]\..*?\}\);",
    "// root route handled by static middleware",
    patched
)
with open('backend/server.js', 'w') as f:
    f.write(patched)
print('✅ Patched server.js — static files served before API routes')

# ── Step 4: Start ngrok tunnel ───────────────────────────────────────────────
ngrok.kill()
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
tunnel = ngrok.connect(5000)
print(f'\n🌐 Public URL: {tunnel.public_url}')

# ── Step 5: Start the backend (serves API + frontend) ───────────────────────
env = {**os.environ, 'GROQ_API_KEY': GROQ_API_KEY, 'PORT': '5000'}
if GITHUB_TOKEN:
    env['GITHUB_TOKEN'] = GITHUB_TOKEN

proc = subprocess.Popen(
    ['node', 'server.js'],
    cwd='backend',
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

time.sleep(2)
print('\n' + '─'*50)
print(f'🚀 CoBug is live!  Open: {tunnel.public_url}')
print('─'*50)
print('Server logs (keep this cell running):')
print()

try:
    for line in proc.stdout:
        print(line, end='')
except KeyboardInterrupt:
    proc.terminate()
    ngrok.kill()
    print('\n🛑 Server stopped')


In [ ]:
# ── Cell 7 (optional): Stop the server ─────────────────────────────────────
# Run this cell only when you want to shut everything down
try:
    proc.terminate()
except:
    pass
from pyngrok import ngrok
ngrok.kill()
print('🛑 Server and ngrok tunnel stopped')